# DecodingExampleWithHist

This notebook demonstrates state decoding with and without history correction.

Workflow:
1. Simulate latent discrete state trajectory with Poisson population activity.
2. Add global history modulation to spike counts.
3. Decode states using Bayes filtering before and after history correction.
4. Validate that history correction improves decoding error.

Reference: DOI `10.1016/j.jneumeth.2012.08.009`, PMID `22981419`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.decoding import DecodingAlgorithms
from nstat.history import HistoryBasis
from nstat.spikes import SpikeTrain

rng = np.random.default_rng(2026)
print('Seed:', 2026)


In [ ]:
n_units = 14
n_states = 15
n_time = 260
dt = 0.02

transition = np.zeros((n_states, n_states), dtype=float)
for i in range(n_states):
    for j, w in ((i - 1, 0.2), (i, 0.6), (i + 1, 0.2)):
        if 0 <= j < n_states:
            transition[i, j] += w
    transition[i, :] /= np.sum(transition[i, :])

latent = np.zeros(n_time, dtype=int)
latent[0] = n_states // 2
for t in range(1, n_time):
    latent[t] = rng.choice(n_states, p=transition[latent[t - 1]])

centers = np.linspace(0.0, n_states - 1, n_units)
widths = np.full(n_units, 2.2)
state_axis = np.arange(n_states)[None, :]
tuning = 0.06 + 0.42 * np.exp(-0.5 * ((state_axis - centers[:, None]) / widths[:, None]) ** 2)

history_weight = 0.55
history_gain = np.ones(n_time, dtype=float)
spike_counts = np.zeros((n_units, n_time), dtype=float)
prev_global = 0.0
for t in range(n_time):
    history_gain[t] = np.exp(history_weight * prev_global)
    lam = tuning[:, latent[t]] * history_gain[t]
    spike_counts[:, t] = rng.poisson(lam)
    prev_global = float(np.mean(spike_counts[:, t]))

decoded_raw, posterior_raw = DecodingAlgorithms.decode_state_posterior(
    spike_counts=spike_counts,
    tuning_rates=tuning,
    transition=transition,
)

corrected_counts = spike_counts / history_gain[None, :]
decoded_hist, posterior_hist = DecodingAlgorithms.decode_state_posterior(
    spike_counts=corrected_counts,
    tuning_rates=tuning,
    transition=transition,
)

nrmse_raw = float(np.sqrt(np.mean((decoded_raw - latent) ** 2)) / (n_states - 1))
nrmse_hist = float(np.sqrt(np.mean((decoded_hist - latent) ** 2)) / (n_states - 1))
print('NRMSE (raw):', nrmse_raw)
print('NRMSE (history-corrected):', nrmse_hist)
print('Posterior normalization max error:', float(np.max(np.abs(np.sum(posterior_hist, axis=0) - 1.0))))

assert nrmse_hist <= nrmse_raw + 1e-8, 'History correction should not degrade decoding.'
assert nrmse_hist < 0.20, f'History-corrected NRMSE too high: {nrmse_hist}'
assert np.allclose(np.sum(posterior_hist, axis=0), 1.0, atol=1e-6)


In [ ]:
time_grid = np.arange(n_time, dtype=float) * dt
spike_times_unit0 = []
for i, n in enumerate(spike_counts[0].astype(int)):
    if n <= 0:
        continue
    t0 = time_grid[i]
    t1 = t0 + dt
    spike_times_unit0.extend(rng.uniform(t0, t1, size=n).tolist())

spike_train = SpikeTrain(
    spike_times=np.sort(np.asarray(spike_times_unit0, dtype=float)),
    t_start=float(time_grid[0]),
    t_end=float(time_grid[-1] + dt),
)
history_basis = HistoryBasis(np.array([0.0, 0.02, 0.06, 0.12]))
H = history_basis.design_matrix(spike_train.spike_times, time_grid)
print('History matrix shape:', H.shape)
assert H.shape == (n_time, history_basis.n_bins)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(latent, label='true state', linewidth=1.2)
axes[0].plot(decoded_raw, label='decoded raw', linewidth=1.0, alpha=0.8)
axes[0].plot(decoded_hist, label='decoded with history correction', linewidth=1.0, alpha=0.9)
axes[0].set_ylabel('state index')
axes[0].set_title('State decoding with history effects')
axes[0].legend(loc='upper right')

axes[1].plot(history_gain, color='tab:purple', linewidth=1.2)
axes[1].set_xlabel('time bin')
axes[1].set_ylabel('gain')
axes[1].set_title('Global history gain factor used in simulation')

plt.tight_layout()
plt.show()
